# CPC Domain-Aware Pretraining on Kaggle

## Changes in this encoder vs previous runs

### Architecture (`model/models.py`)
1. **GroupNorm(8, 256) replaces BatchNorm1d** in `g_enc` (both `CPCClassifier` and `CPCModel`)
   - BatchNorm computes stats across the batch — if a batch mixes devices with different amplitude scales, it blends their statistics and leaks domain info into every sample
   - GroupNorm computes stats per-sample (8 groups of 32 channels), so it's domain-agnostic by construction

2. **Spectral normalization on both CausalConv1d layers** in `g_enc`
   - Constrains the largest singular value of each weight matrix to 1 (Lipschitz constraint)
   - Guarantees small input perturbations (domain shifts) produce proportionally small representation changes
   - Prevents the encoder from amplifying domain-specific noise into large feature differences

### Pretraining (`engine/cpc_trainer.py`)
3. **CSI augmentation during CPC pretraining** (previously only used in masked pretraining)
   - Amplitude scaling (0.3x–3.0x) — simulates large cross-device gain differences
   - Gaussian noise (std=0.15, 50% prob) — simulates environmental interference
   - Time shift (up to 50 steps, 50% prob)
   - Subcarrier dropout (up to 30, 50% prob) — simulates devices with fewer antennas
   - Frequency jitter (std=0.1, 30% prob) — simulates device frequency offsets
   - **Why:** contrastive SSL learns invariance to whatever augmentations simulate. Previous CPC had zero augmentation.

4. **Domain-aware hard negative linear ramp** (0.0 → 0.8)
   - Early epochs: all negatives random (easy) — encoder learns basic temporal structure
   - Late epochs: 80% negatives from same (env, device) domain — forces encoder to distinguish content from domain
   - Previously was a fixed ratio of 0.5; curriculum approach is more stable and reaches higher hardness

5. **No early stopping** — runs full epoch budget. NaN detection stops + restores best checkpoint if gradient explosion occurs.

### CLI (`scripts/pretrain.py`)
6. **`--pretrain_method cpc_domain_aware`** as a standalone option (auto-sets `domain_aware=True`)

---

## How to run this notebook on Kaggle

1. Go to [kaggle.com](https://www.kaggle.com) → **Code** → **New Notebook**
2. On the right sidebar, click **Settings** → **Accelerator** → select **GPU T4 x2**
3. On the right sidebar, click **Input** → **Add Input** → search `csi-bench` → add the dataset by guozhenjennzhu
4. Upload this `.ipynb` file: **File** → **Import Notebook** → select `kaggle_pretrain.ipynb`
5. Update cell 1 below with your GitHub username
6. Click **Run All** (or run cells one at a time)
7. When done, download `encoder_weights_domain_aware_aug.pt` from the **Output** tab on the right sidebar
8. Finetune locally:
```
python scripts/run_experiment_trials.py \
  --encoder encoder_weights_domain_aware_aug.pt \
  --tasks all --tag domain_aware_aug_v2
```

In [ ]:
# 1. Clone your repo
!git clone https://github.com/YOUR_USERNAME/csi-bench-SSL.git /kaggle/working/csi-bench-SSL
%cd /kaggle/working/csi-bench-SSL
!git checkout DANN  # your working branch

In [ ]:
# 2. Install deps (most are pre-installed on Kaggle)
!pip install -q pyaml tqdm scikit-learn seaborn h5py

In [ ]:
# 3. Symlink the Kaggle dataset into the expected data/ directory
import os
os.makedirs('data', exist_ok=True)

# The CSI-Bench dataset on Kaggle is at /kaggle/input/csi-bench/
# Check the actual structure:
!ls /kaggle/input/csi-bench/

In [ ]:
# Symlink each task directory into data/
# Adjust paths if the Kaggle dataset has a different structure
import glob

kaggle_data = '/kaggle/input/csi-bench'
tasks = ['BreathingDetection', 'FallDetection', 'Localization', 'MotionSourceRecognition',
         'HumanActivityRecognition', 'HumanIdentification', 'ProximityRecognition']

for task in tasks:
    src = os.path.join(kaggle_data, task)
    dst = os.path.join('data', task)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'Linked {src} -> {dst}')
    elif not os.path.exists(src):
        # Maybe nested under a subdirectory
        candidates = glob.glob(f'/kaggle/input/csi-bench/*/{task}')
        if candidates:
            os.symlink(candidates[0], dst)
            print(f'Linked {candidates[0]} -> {dst}')
        else:
            print(f'WARNING: {task} not found in Kaggle dataset')

!ls -la data/

In [ ]:
# 4. Run domain-aware CPC pretraining with augmentations
# - GroupNorm + spectral norm in g_enc (architecture changes)
# - CSI augmentation (amp scaling, noise, subcarrier dropout, freq jitter)
# - Domain-aware hard negative ramp: 0.0 -> 0.8 over training
!python scripts/pretrain.py \
    --pretrain_method cpc_domain_aware \
    --all_tasks \
    --domain_neg_ratio 0.8 \
    --epochs 300 \
    --batch_size 32 \
    --learning_rate 3e-4 \
    --save_dir pretrain_results

In [ ]:
# 5. Check the training curve
import pandas as pd
import matplotlib.pyplot as plt

results_dirs = !ls -d pretrain_results/all_tasks/cpc/*/
latest = sorted(results_dirs)[-1]
print(f'Results dir: {latest}')

df = pd.read_csv(f'{latest}/training_results.csv')
plt.figure(figsize=(10, 5))
plt.plot(df['Epoch'], df['Train Loss'])
plt.xlabel('Epoch')
plt.ylabel('Train Loss')
plt.title('CPC Domain-Aware Pretraining (with augmentation)')
plt.grid(True)
plt.show()

print(f"Final loss: {df['Train Loss'].iloc[-1]:.4f}")
print(f"Best loss: {df['Train Loss'].min():.4f} at epoch {df.loc[df['Train Loss'].idxmin(), 'Epoch']}")

In [ ]:
# 6. Save the encoder weights — download this from Kaggle output
import shutil
encoder_path = f'{latest}/encoder_weights.pt'
output_path = '/kaggle/working/encoder_weights_domain_aware_aug.pt'
shutil.copy(encoder_path, output_path)
print(f'Encoder saved to {output_path}')
print('Download this file from the Output tab on the right sidebar.')

In [ ]:
# 7. (Optional) Also run standard CPC pretraining for comparison
!python scripts/pretrain.py \
    --pretrain_method cpc \
    --all_tasks \
    --epochs 300 \
    --batch_size 32 \
    --learning_rate 3e-4 \
    --save_dir pretrain_results